# 02 - Detection, Segmentation, Video, and Defect Blueprint

This notebook gives you practical templates for real inspection workflows.

## Scope
- Object detection baseline
- Segmentation-style mask baseline
- Video frame processing and anomaly scoring
- Enterprise defect detection architecture for phone transit


## 0) Dataset and test-file download references

Use these links to fetch data before your flight when possible:
- MVTec AD: https://www.mvtec.com/company/research/datasets/mvtec-ad
- COCO: https://cocodataset.org/
- Penn-Fudan: https://www.cis.upenn.edu/~jshi/ped_html/
- Sample videos: https://www.pexels.com/videos/

Place your local files into:
- `data/images/`
- `data/video/`


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import torch
import timm
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
ARTIFACTS_DIR = ROOT / 'artifacts'
VIDEO_DIR = DATA_DIR / 'video'
IMG_DIR = DATA_DIR / 'images'

for p in [DATA_DIR, ARTIFACTS_DIR, VIDEO_DIR, IMG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


## 1) Load DINO backbone and preprocessing
This embedding model supports downstream scoring and weak localization.


In [ ]:
model_name = 'vit_base_patch14_dinov2.lvd142m'
if model_name not in set(timm.list_models(pretrained=True)):
    model_name = 'vit_small_patch14_dinov2.lvd142m'

backbone = timm.create_model(model_name, pretrained=True, num_classes=0).eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Loaded backbone:', model_name)


## 2) Minimal object detection baseline
For speed, we use a pre-trained detector from torchvision as a practical baseline.
In production, swap to your custom detector with DINO-based features.


In [ ]:
import torchvision

detector = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
detector.eval().to(device)
print('Detection model loaded')

def detect_objects(pil_img, score_thr=0.6):
    x = transforms.ToTensor()(pil_img).to(device)
    with torch.no_grad():
        out = detector([x])[0]
    keep = out['scores'].detach().cpu().numpy() >= score_thr
    boxes = out['boxes'].detach().cpu().numpy()[keep]
    scores = out['scores'].detach().cpu().numpy()[keep]
    labels = out['labels'].detach().cpu().numpy()[keep]
    return boxes, scores, labels


## 3) Simple segmentation baseline
This gives a segmentation workflow template (semantic mask output).


In [ ]:
seg_model = torchvision.models.segmentation.deeplabv3_resnet50(weights='DEFAULT').eval().to(device)
print('Segmentation model loaded')

def segment_image(pil_img):
    x = transforms.ToTensor()(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = seg_model(x)['out']
    mask = out.argmax(dim=1).squeeze(0).detach().cpu().numpy().astype(np.uint8)
    return mask


## 4) DINO embedding helper for anomaly scoring
We compute distance from a reference "healthy" embedding center.


In [ ]:
def dino_embed_pil(pil_img):
    x = preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = backbone(x).squeeze(0).detach().cpu().numpy()
    return emb

def cosine_distance(a, b, eps=1e-8):
    return 1.0 - (np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps))


## 5) Video processing pipeline

Place a local video in `data/video/` and update `video_path`.
This script samples frames, runs DINO embedding extraction, and computes anomaly score
relative to the first N frames assumed healthy.


In [ ]:
video_files = list(VIDEO_DIR.glob('*.mp4')) + list(VIDEO_DIR.glob('*.mov')) + list(VIDEO_DIR.glob('*.avi'))
if not video_files:
    print('No video file found in', VIDEO_DIR)
else:
    video_path = str(video_files[0])
    print('Using video:', video_path)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = 0
    sample_every = max(int(fps // 2), 1)  # about 2 samples/sec
    embs = []
    sampled_indices = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_every == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil = Image.fromarray(rgb)
            embs.append(dino_embed_pil(pil))
            sampled_indices.append(frame_idx)
        frame_idx += 1
    cap.release()

    embs = np.array(embs)
    print('Sampled frames:', len(embs))

    if len(embs) > 10:
        ref_center = embs[:10].mean(axis=0)
        scores = np.array([cosine_distance(e, ref_center) for e in embs])

        plt.figure(figsize=(10, 4))
        plt.plot(sampled_indices, scores)
        plt.title('Frame-level DINO Anomaly Score')
        plt.xlabel('Frame Index')
        plt.ylabel('Cosine Distance from Healthy Reference')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        np.save(ARTIFACTS_DIR / 'video_anomaly_scores.npy', scores)
        np.save(ARTIFACTS_DIR / 'video_sampled_frame_idx.npy', np.array(sampled_indices))
        print('Saved anomaly artifacts to', ARTIFACTS_DIR)


## 6) Defect detection system blueprint (enterprise)

Use this staged approach:
1. **Input quality gate**: reject blurry/dark frames.
2. **Localization**: detect package + phone regions.
3. **Fine analysis**: segmentation/part-specific inspection.
4. **DINO anomaly score**: compare embeddings with known-good references.
5. **Fusion**: weighted score + threshold calibration by business risk.
6. **Review queue**: route uncertain cases to human QA.

This directly maps to a production defect pipeline for phones in transit.


## 7) Immediate next steps
- Replace public data with your domain images/videos.
- Label 4-8 defect classes and at least 300-1000 examples/class.
- Start with frozen DINO features + linear/MLP heads.
- Move to end-to-end fine-tuning only after baseline stabilization.
